In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import xgboost as xgb

import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)


In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import xgboost as xgb

import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)


In [3]:
# ==== CONFIG: EDIT THIS PART ====

# Path to your Kaggle dataset CSV file
DATA_PATH = "Product_Sales_Dataset_2023_2024.csv"  # change to your actual file name

# Column names in the raw data (adjust to real names)
DATE_COL       = "Order Date"      # or "Date"
PRODUCT_COL    = "Product Name"    # or "Product", "Product_ID"
SALES_COL      = "Sales"           # or "Total_Sales", "Revenue"
REGION_COL     = "Region"          # or "Country", "State", etc.
CATEGORY_COL   = "Category"        # or "Product_Category", etc.

# Granularity assumptions:
# If your data is at the ORDER level (multiple rows per product per day),
# we will aggregate to daily product-level sales.
AGGREGATE_TO_DAILY = True

# Forecast horizon (in days) beyond the last date in the dataset
FORECAST_HORIZON_DAYS = 30

# ================================


In [5]:
df_raw = pd.read_csv("/content/product_sales_dataset_final.csv")
print("Shape:", df_raw.shape)
df_raw.head()


Shape: (200000, 14)


,Order_ID,Order_Date,Customer_Name,City,State,Region,Country,Category,Sub_Category,Product_Name,Quantity,Unit_Price,Revenue,Profit
0,1,08-23-23,Bianca Brown,Jackson,Mississippi,South,United States,Accessories,Small Electronics,Phone Case,3,201.01,603.03,221.49
1,2,12-20-24,Jared Edwards,Grand Rapids,Michigan,Centre,United States,Accessories,Small Electronics,Charging Cable,4,74.30,297.20,97.09
2,3,01-29-24,Susan Valdez,Minneapolis,Minnesota,Centre,United States,Clothing & Apparel,Sportswear,Nike Air Force 1,1,68.19,68.19,25.47
3,4,11-29-24,Tina Williams,Tallahassee,Florida,South,United States,Clothing & Apparel,Sportswear,Adidas Tracksuit,3,209.64,628.92,231.38
4,5,09-21-23,Catherine Gordon,Baltimore,Maryland,East,United States,Accessories,Bags,Backpack,1,216.63,216.63,42.46


In [6]:
df_raw.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 14 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Order_ID       200000 non-null  int64  
 1   Order_Date     200000 non-null  object 
 2   Customer_Name  200000 non-null  object 
 3   City           200000 non-null  object 
 4   State          200000 non-null  object 
 5   Region         200000 non-null  object 
 6   Country        200000 non-null  object 
 7   Category       200000 non-null  object 
 8   Sub_Category   200000 non-null  object 
 9   Product_Name   200000 non-null  object 
 10  Quantity       200000 non-null  int64  
 11   Unit_Price    200000 non-null  float64
 12   Revenue       200000 non-null  float64
 13   Profit        200000 non-null  float64
dtypes: float64(3), int64(2), object(9)
memory usage: 21.4+ MB


In [8]:
# ====== STEP 5: Aggregate to Daily Product Sales ======

df = df_raw.copy()

# Clean column names (optional but recommended)
df.columns = df.columns.str.strip()   # removes spaces like " Revenue "
df.rename(columns={"Unit_Price": "Unit_Price"}, inplace=True)

DATE_COL       = "Order_Date"
PRODUCT_COL    = "Product_Name"
SALES_COL      = "Revenue"
REGION_COL     = "Region"
CATEGORY_COL   = "Category"

# Convert to datetime
df[DATE_COL] = pd.to_datetime(df[DATE_COL])

# Aggregate to daily product-level sales
group_cols = [DATE_COL, PRODUCT_COL]

# Your dataset has Region + Category → include them
group_cols.append(REGION_COL)
group_cols.append(CATEGORY_COL)

df_daily = (
    df.groupby(group_cols, as_index=False)[SALES_COL].sum()
)

df_daily = df_daily.sort_values([PRODUCT_COL, DATE_COL]).reset_index(drop=True)

df_daily.head()


,Order_Date,Product_Name,Region,Category,Revenue
0,2023-01-01,Adidas Tracksuit,Centre,Clothing & Apparel,1618.31
1,2023-01-01,Adidas Tracksuit,East,Clothing & Apparel,224.82
2,2023-01-01,Adidas Tracksuit,West,Clothing & Apparel,1020.36
3,2023-01-02,Adidas Tracksuit,Centre,Clothing & Apparel,198.41
4,2023-01-02,Adidas Tracksuit,South,Clothing & Apparel,846.06
